# 02 — Feature Engineering
**Projet** : ObRail MSPR 2025-2026  

**Source** : `data/processed/routes_processed.csv`  
**Résultat** : `data/processed/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`  

**Objectif** : Construire le feature set final, encoder les variables catégorielles, normaliser les numériques, et produire les splits train/test utilisés par tous les scripts de modélisation.

## 0. Imports et configuration

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for folder in [cwd] + list(cwd.parents):
        if (folder / 'data').exists() and (folder / 'src').exists() and (folder / 'models').exists():
            return folder
    raise FileNotFoundError('Could not find project root from current working directory')

ROOT = find_project_root()
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'is_underserved'

print('✅ Imports OK')

✅ Imports OK


## 1. Chargement des données

In [2]:
df = pd.read_csv(PROCESSED_DIR / 'routes_processed.csv', dtype={'days_of_week': str})
print(f'Shape : {df.shape}')
df.head()

Shape : (25200, 21)


,agency_name,route_id,route_type,route_short_name,route_long_name,departure_country,arrival_country,days_of_week,is_night_train,distance_km,...,emissions_co2,service_type,days_active,is_underserved,rail_modal_share,type_encoded,is_international,log_distance,log_co2,country_encoded
0,"Železničná Spoločnosť Slovensko, A.s.",107,2,NaN,bratislava-n.mesto-nitra,AT,AT,1111111.0,False,97.31,...,2123.012,day,7,0,12.1,0,0,4.588126,3.127506,4
1,"Železničná Spoločnosť Slovensko, A.s.",13,2,NaN,budapest-nyugati pu-hamburg-altona,AT,AT,1100000.0,False,258.62,...,5642.313,day,2,1,12.1,0,0,5.559219,3.127506,4
2,"Železničná Spoločnosť Slovensko, A.s.",97,2,NaN,červená skala-banská bystrica,AT,AT,1111100.0,False,81.61,...,1780.485,day,5,0,12.1,0,0,4.414131,3.127506,4
3,"Železničná Spoločnosť Slovensko, A.s.",104,2,NaN,margecany-banská bystrica,AT,AT,1111111.0,False,150.43,...,3281.931,day,7,0,12.1,0,0,5.020123,3.127506,4
4,"Železničná Spoločnosť Slovensko, A.s.",364,2,NaN,košice-banská bystrica,AT,AT,1111111.0,False,180.49,...,3937.750,day,7,0,12.1,0,0,5.201201,3.127506,4


## 2. Suppression des colonnes sources de la cible

`is_underserved` est construit à partir de la règle :  
`days_active <= 3 AND distance_km > 100`

Ces deux colonnes sont donc des **fuites de données** directes — le modèle apprendrait la règle de construction plutôt que des patterns réels. Elles sont supprimées du feature set.

`log_distance` est conservée avec un caveat documenté : elle encode partiellement le seuil `distance_km > 100` mais capture aussi une réalité opérationnelle légitime (les longues routes sont structurellement plus difficiles à desservir fréquemment). Sa contribution sera vérifiée via SHAP après entraînement.

In [3]:
LEAKY_COLS = ['days_active', 'distance_km']

# Colonnes non utilisées comme features (métadonnées textuelles ou identifiants)
META_COLS = [
    'agency_name', 'route_id', 'route_type',
    'route_short_name', 'route_long_name',
    'days_of_week', 'is_night_train',
    'arrival_country',
    'service_type',       # remplacée par type_encoded
    'departure_country',  # sera encodée en one-hot ci-dessous
    'co2_per_pkm',        # remplacée par log_co2
    'emissions_co2',      # non retenue (redondante avec log_co2)
    'country_encoded',    # remplacé par one-hot
]

cols_to_drop = LEAKY_COLS + META_COLS
print(f'Colonnes supprimées : {cols_to_drop}')

Colonnes supprimées : ['days_active', 'distance_km', 'agency_name', 'route_id', 'route_type', 'route_short_name', 'route_long_name', 'days_of_week', 'is_night_train', 'arrival_country', 'service_type', 'departure_country', 'co2_per_pkm', 'emissions_co2', 'country_encoded']


## 3. Encodage one-hot de `departure_country`

Le pays de départ est une variable catégorielle nominale — il n'existe pas d'ordre naturel entre les pays. Un encodage ordinal (FR=0, DE=1...) introduirait une relation numérique artificielle. On utilise donc le **one-hot encoding** : une colonne binaire par pays.

Avec 34 pays distincts, cela ajoute 34 colonnes. `drop='first'` supprime une colonne pour éviter la multicolinéarité parfaite (problème surtout pour la régression logistique).

In [4]:
country_dummies = pd.get_dummies(
    df['departure_country'],
    prefix='country',
    drop_first=True,
    dtype=int,
)

print(f'Colonnes one-hot créées : {country_dummies.shape[1]}')
print(country_dummies.columns.tolist())

Colonnes one-hot créées : 33
['country_AT', 'country_BE', 'country_BG', 'country_CH', 'country_CZ', 'country_DE', 'country_DK', 'country_EE', 'country_ES', 'country_FI', 'country_FR', 'country_GB', 'country_GR', 'country_HR', 'country_HU', 'country_IE', 'country_IT', 'country_LT', 'country_LU', 'country_MD', 'country_ME', 'country_MK', 'country_NL', 'country_NO', 'country_PL', 'country_PT', 'country_RO', 'country_RS', 'country_SE', 'country_SI', 'country_SK', 'country_TR', 'country_UA']


## 4. Construction du feature set final

In [5]:
# Features numériques retenues
NUMERIC_FEATURES = [
    'rail_modal_share',   # part modale ferroviaire du pays — signal structurel
    'type_encoded',       # 0=jour, 1=nuit — fort discriminant
    'is_international',   # route internationale — signal très fort (94.4% underserved)
    'log_distance',       # distance log-transformée — caveat documenté section 2
    'log_co2',            # empreinte carbone log-transformée
]

# Assemblage : numériques + one-hot pays
X = pd.concat(
    [df[NUMERIC_FEATURES], country_dummies],
    axis=1,
)
y = df[TARGET]

print(f'Feature set final : {X.shape[1]} colonnes, {X.shape[0]} lignes')
print(f'\nFeatures numériques : {NUMERIC_FEATURES}')
print(f'Features one-hot    : {country_dummies.shape[1]} colonnes pays')
print(f'\nValeurs manquantes  : {X.isnull().sum().sum()}')
X.head()

Feature set final : 38 colonnes, 25200 lignes

Features numériques : ['rail_modal_share', 'type_encoded', 'is_international', 'log_distance', 'log_co2']
Features one-hot    : 33 colonnes pays

Valeurs manquantes  : 0


,rail_modal_share,type_encoded,is_international,log_distance,log_co2,country_AT,country_BE,country_BG,country_CH,country_CZ,...,country_NO,country_PL,country_PT,country_RO,country_RS,country_SE,country_SI,country_SK,country_TR,country_UA
0,12.1,0,0,4.588126,3.127506,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,12.1,0,0,5.559219,3.127506,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,12.1,0,0,4.414131,3.127506,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,12.1,0,0,5.020123,3.127506,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,12.1,0,0,5.201201,3.127506,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 4.1 Tableau des variables retenues

Livrable explicitement demandé par le cahier des charges ObRail (Besoin 1).

In [6]:
variables_table = pd.DataFrame([
    {
        'Variable': 'rail_modal_share',
        'Type': 'Numérique continu',
        'Source': 'Eurostat (carte pays)',
        'Justification': 'Part modale ferroviaire du pays de départ — proxy du niveau d\'investissement réseau',
        'Corrélation cible': 0.09,
    },
    {
        'Variable': 'type_encoded',
        'Type': 'Binaire',
        'Source': 'is_night_train (encodé)',
        'Justification': 'Trains de nuit sous-desservis à 75.4% vs 17.9% pour les trains de jour',
        'Corrélation cible': 0.23,
    },
    {
        'Variable': 'is_international',
        'Type': 'Binaire',
        'Source': 'Comparaison departure/arrival country',
        'Justification': 'Routes internationales sous-desservies à 94.4% vs 18.5% pour domestiques',
        'Corrélation cible': 0.20,
    },
    {
        'Variable': 'log_distance',
        'Type': 'Numérique continu',
        'Source': 'log(1 + distance_km)',
        'Justification': 'Distance opérationnelle — corrélation 0.48 dont partie liée au seuil 100km de la règle cible. Vérifiée via SHAP.',
        'Corrélation cible': 0.48,
    },
    {
        'Variable': 'log_co2',
        'Type': 'Numérique continu',
        'Source': 'log(1 + co2_per_pkm)',
        'Justification': 'Empreinte carbone par passager-km — indicateur de type de traction et d\'efficacité',
        'Corrélation cible': -0.01,
    },
    {
        'Variable': 'country_XX (×33)',
        'Type': 'Binaire (one-hot)',
        'Source': 'departure_country (encodé)',
        'Justification': 'Effet pays — disparités importantes observées en EDA (NL 37.6%, SI 1.7%)',
        'Corrélation cible': 'variable',
    },
])

print('Tableau des variables retenues :')
display(variables_table)

# Sauvegarde du tableau
variables_table.to_csv(PROCESSED_DIR / 'variables_retenues.csv', index=False)
print('\n✅ Sauvegardé → data/processed/variables_retenues.csv')

Tableau des variables retenues :


,Variable,Type,Source,Justification,Corrélation cible
0,rail_modal_share,Numérique continu,Eurostat (carte pays),Part modale ferroviaire du pays de départ — pr...,0.09
1,type_encoded,Binaire,is_night_train (encodé),Trains de nuit sous-desservis à 75.4% vs 17.9%...,0.23
2,is_international,Binaire,Comparaison departure/arrival country,Routes internationales sous-desservies à 94.4%...,0.2
3,log_distance,Numérique continu,log(1 + distance_km),Distance opérationnelle — corrélation 0.48 don...,0.48
4,log_co2,Numérique continu,log(1 + co2_per_pkm),Empreinte carbone par passager-km — indicateur...,-0.01
5,country_XX (×33),Binaire (one-hot),departure_country (encodé),Effet pays — disparités importantes observées ...,variable



✅ Sauvegardé → data/processed/variables_retenues.csv


## 5. Split train/test

**Ratio choisi : 80/20**
- 80% entraînement (~20 160 routes) : suffisant pour que les modèles apprennent
- 20% test (~5 040 routes) : assez large pour une évaluation robuste
- La cross-validation (5-fold) sera utilisée lors du tuning — pas besoin d'un set de validation séparé

**`stratify=y`** : garantit que le ratio 80.6% / 19.4% est respecté dans les deux splits — indispensable avec un déséquilibre de classes.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'\nDistribution y_train :')
print(y_train.value_counts())
print(f'Taux underserved train : {y_train.mean()*100:.1f}%')
print(f'\nDistribution y_test :')
print(y_test.value_counts())
print(f'Taux underserved test  : {y_test.mean()*100:.1f}%')

X_train : (20160, 38)
X_test  : (5040, 38)

Distribution y_train :
is_underserved
0    16250
1     3910
Name: count, dtype: int64
Taux underserved train : 19.4%

Distribution y_test :
is_underserved
0    4063
1     977
Name: count, dtype: int64
Taux underserved test  : 19.4%


## 6. Normalisation des features numériques

**Pourquoi normaliser ?**
- La régression logistique et le MLP sont sensibles à l'échelle des features — `log_distance` va de 0.2 à 8.9, `rail_modal_share` de 1.9 à 17.9. Sans normalisation, les features à grande échelle dominent.
- LightGBM et RandomForest n'en ont pas besoin (modèles à base d'arbres) mais la normalisation ne les affecte pas non plus.
- On normalise donc tout le feature set pour avoir un pipeline unifié.

**Important** : le scaler est entraîné **uniquement sur X_train** puis appliqué à X_test. Entraîner sur l'ensemble complet introduirait une fuite d'information du test vers le train.

In [8]:
# Colonnes à normaliser — uniquement les numériques continues
# Les binaires (type_encoded, is_international, country_XX) ne sont pas normalisées
COLS_TO_SCALE = ['rail_modal_share', 'log_distance', 'log_co2']

scaler = StandardScaler()

X_train = X_train.copy()
X_test  = X_test.copy()

X_train[COLS_TO_SCALE] = scaler.fit_transform(X_train[COLS_TO_SCALE])
X_test[COLS_TO_SCALE]  = scaler.transform(X_test[COLS_TO_SCALE])

print('Statistiques après normalisation (X_train) :')
display(X_train[COLS_TO_SCALE].describe().round(4))

print('\nStatistiques X_test (appliqué avec le scaler du train) :')
display(X_test[COLS_TO_SCALE].describe().round(4))

Statistiques après normalisation (X_train) :


,rail_modal_share,log_distance,log_co2
count,20160.0000,20160.0000,20160.0000
mean,0.0000,-0.0000,0.0000
std,1.0000,1.0000,1.0000
min,-2.9968,-5.1748,-5.7897
25%,-0.9992,-0.7363,-0.0652
50%,0.4772,-0.2907,0.0343
75%,0.9983,0.6064,0.2444
max,3.9513,4.6894,2.1365



Statistiques X_test (appliqué avec le scaler du train) :


,rail_modal_share,log_distance,log_co2
count,5040.0000,5040.0000,5040.0000
mean,0.0085,0.0278,0.0231
std,1.0052,1.0026,0.9568
min,-2.9100,-4.1985,-5.6163
25%,-0.9992,-0.7310,-0.0652
50%,0.4772,-0.2491,0.0343
75%,0.9983,0.6690,0.2808
max,3.9513,4.6526,2.1365


## 7. Sauvegarde des splits et du scaler

In [9]:
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR  / 'X_test.csv',  index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv(PROCESSED_DIR  / 'y_test.csv',  index=False)

joblib.dump(scaler, PROCESSED_DIR / 'scaler.joblib')

print('✅ Splits sauvegardés :')
print(f'   X_train : {X_train.shape}')
print(f'   X_test  : {X_test.shape}')
print(f'   y_train : {y_train.shape}')
print(f'   y_test  : {y_test.shape}')
print('✅ Scaler sauvegardé → data/processed/scaler.joblib')

✅ Splits sauvegardés :
   X_train : (20160, 38)
   X_test  : (5040, 38)
   y_train : (20160,)
   y_test  : (5040,)
✅ Scaler sauvegardé → data/processed/scaler.joblib


## 8. Récapitulatif

**Ce qui a été fait dans ce notebook**
- Suppression des colonnes sources de `is_underserved` (`days_active`, `distance_km`)
- Encodage one-hot de `departure_country` (33 colonnes, `drop_first=True`)
- Feature set final : 5 numériques + 33 one-hot = **38 features**
- Split stratifié 80/20 — ratio underserved préservé dans les deux splits
- Normalisation StandardScaler sur les 3 features continues (`rail_modal_share`, `log_distance`, `log_co2`) — scaler entraîné sur train uniquement
- Sauvegarde de `X_train`, `X_test`, `y_train`, `y_test`, `scaler.joblib`, `variables_retenues.csv`

**Prochaine étape** : `03_models.ipynb` — entraînement et comparaison des modèles candidats sur ces splits.